# K-Index: Mechanism Tests & Advisor Regressions

Two rounds of advisor feedback, addressed in order:

1. **Girand-adjacent ask:** does K explain asset prices (stock index, 10Y Treasury total
   return, USD/JPY, USD/EUR, USD/GBP, gold) and economic growth (unemployment, GDP,
   industrial production)? Each as `variable_t ~ const + K_t + K_(t-1..4)`, HAC errors.
2. **Round 2 feedback:** *"I would expect to see widening → stocks go up, tightening →
   stocks go down. Widening → consumer credit worsens, tightening → consumer credit
   improves... Professor Melvin mentioned the effects on assets might be lagged. No
   relationship is incredibly useful (Buffett wouldn't expect a relationship!)."*

That second round reframes the test: **contemporaneous first** (the mechanical wealth-effect
channel — equity rallies concentrate gains among top-wealth holders, pushing K up the *same*
quarter), **lagged second** (Melvin's addendum). This notebook keeps those two views separate
rather than one combined regression, and tests consumer credit and an equity-growth proxy with
data already available — no blocked external data needed for those two.

This notebook rebuilds K from scratch (same method as `01_KIndex_Build_and_Validate.ipynb`) so
it runs standalone.

## 0 · Setup — rebuild K

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from pathlib import Path

pd.set_option("display.width", 120)
ROOT = Path("..")
OUT = Path("output"); OUT.mkdir(exist_ok=True)
CACHE_DIR = Path("data_cache"); CACHE_DIR.mkdir(exist_ok=True)

def dfa_quarter(q):
    y, qq = str(q).split(":Q"); return pd.Period(f"{y}Q{qq}", freq="Q")

def to_qend(s):
    s = s.copy(); s.index = pd.to_datetime(s.index).to_period("Q").to_timestamp("Q")
    return s.groupby(s.index).mean()

def zscore(s):
    return (s - s.mean()) / s.std(ddof=0)

def parse_dfa_date(s):
    return pd.PeriodIndex(s.str.replace(":", "-"), freq="Q").to_timestamp(how="end").normalize()

# Wealth
det = pd.read_csv(ROOT / "dfa-networth-levels-detail.csv"); det["t"] = det["Date"].map(dfa_quarter)
nw = det.pivot_table(index="t", columns="Category", values="Net worth", aggfunc="sum")
hh = det.pivot_table(index="t", columns="Category", values="Household count", aggfunc="sum")
top, bot = ["TopPt1","RemainingTop1","Next9"], ["Bottom50"]
wealth = (nw[top].sum(axis=1)/hh[top].sum(axis=1)) / (nw[bot].sum(axis=1)/hh[bot].sum(axis=1))
wealth.index = wealth.index.to_timestamp(how="end").normalize()
wealth = to_qend(wealth.rename("wealth"))

# Income
w = pd.read_excel(ROOT / "wage-growth-data.xlsx", sheet_name="Average Wage Quartile", header=None).iloc[3:].copy()
w.columns = ["date","q1","q2","q3","q4","overall","low_half","up_half"]
w["date"] = pd.to_datetime(w["date"])
for c in w.columns[1:]:
    w[c] = pd.to_numeric(w[c].replace(".", np.nan), errors="coerce")
w = w.set_index("date").dropna(subset=["q1","q4"], how="all")
income = to_qend((w["q4"] - w["q1"]).resample("QE").mean().rename("income"))

# Consumer
m = pd.read_excel(ROOT / "bluebk02n.xls", sheet_name=0, header=None).iloc[8:].copy()
m.columns = ["qlabel","year","sent_bot","sent_mid","sent_top","cur_bot","cur_mid","cur_top","exp_bot","exp_mid","exp_top"]
m = m[m["year"].notna()].copy()
qmap = {"Jan.-Mar.":1, "Apr.-Jun.":2, "Jul.-Sep.":3, "Oct.-Dec.":4}
m["q"] = m["qlabel"].astype(str).str.strip().map(qmap)
m = m[m["q"].notna()].copy()
m["t"] = [pd.Period(f"{int(y)}Q{int(q)}", freq="Q").to_timestamp("Q") for y, q in zip(m["year"], m["q"])]
m = m.set_index("t")
m["sent_bot"] = pd.to_numeric(m["sent_bot"], errors="coerce")
m["sent_top"] = pd.to_numeric(m["sent_top"], errors="coerce")
consumer = to_qend((m["sent_top"] - m["sent_bot"]).rename("consumer"))

# Assemble
common = pd.concat([wealth, income, consumer], axis=1, sort=True).dropna().copy()
common["wealth"] = np.log(common["wealth"])
Z = common.apply(zscore)
Z.columns = ["z_wealth", "z_income", "z_consumer"]
Z["K"] = Z.mean(axis=1)
K = Z["K"]

print(f"K rebuilt: {K.index.min().date()} -> {K.index.max().date()} ({len(K)} quarters)")
print(f"Latest K: {K.dropna().iloc[-1]:.4f} ({K.dropna().index[-1].date()})")


K rebuilt: 1997-12-31 -> 2026-03-31 (114 quarters)
Latest K: 0.0666 (2026-03-31)


## 1 · Consumer credit stress — real data, no blocked dependency

`dfa-networth-levels-detail.csv` (already used above) has a real dollar-level "Consumer
credit" column by wealth percentile. Bottom 50%'s consumer credit as a share of their own
assets is a direct leverage/stress proxy: rising = more debt-financed, i.e. "worsening" in the
advisor's framing.

In [2]:
df = pd.read_csv(ROOT / "dfa-networth-levels-detail.csv")
df["Date"] = parse_dfa_date(df["Date"])
bottom = df[df["Category"] == "Bottom50"].set_index("Date").sort_index()
bottom50_leverage = (bottom["Consumer credit"] / bottom["Assets"]).rename("bottom50_credit_leverage")
bottom50_leverage_chg = bottom50_leverage.diff().rename("bottom50_leverage_qoq_chg")

print(bottom50_leverage.tail(5))


Date
2025-03-31    0.259345
2025-06-30    0.255762
2025-09-30    0.254446
2025-12-31    0.253968
2026-03-31    0.253803
Name: bottom50_credit_leverage, dtype: float64


## 2 · Equity growth proxy — real data, no blocked dependency

Stock prices themselves are blocked in this environment (same Yahoo Finance/FRED wall as
`run_k_regressions.py` below). In the meantime, QoQ growth in total household-sector equity
holdings (summed across all 4 generations in `dfa-generation-levels-detail.csv`) is a real,
data-grounded stand-in — it conflates price return with net contribution/withdrawal flows, so
treat it as directional, not a clean total-return series.

In [3]:
EQUITY_COL = "Corporate equities and mutual fund shares"
gen = pd.read_csv(ROOT / "dfa-generation-levels-detail.csv")
gen["Date"] = parse_dfa_date(gen["Date"])
agg_equity_growth = gen.groupby("Date")[EQUITY_COL].sum().sort_index().pct_change().rename("agg_equity_qoq_growth")

print(agg_equity_growth.tail(5))


Date
2025-03-31   -0.019098
2025-06-30    0.080628
2025-09-30    0.075549
2025-12-31    0.025059
2026-03-31   -0.023807
Name: agg_equity_qoq_growth, dtype: float64


## 3 · Contemporaneous vs. lagged regression, kept separate

The advisor's hypothesis is contemporaneous first ("widening → stocks go up", the mechanical
channel), lagged second (Melvin's addendum). Lumping current K and its own lags into one
regression lets multicollinearity between them obscure which one (if either) is doing the
work — the same failure mode BEDI's structural-break test hit earlier in this project. So two
separate regressions, not one combined table.

In [4]:
def contemporaneous_and_lagged_test(target, k, n_lags=4, hac_lags=3):
    df = pd.DataFrame({"y": target, "K": k})
    for lag in range(1, n_lags + 1):
        df[f"K_lag{lag}"] = df["K"].shift(lag)

    contemp_df = df[["y", "K"]].dropna()
    contemp_X = sm.add_constant(contemp_df[["K"]])
    contemp_model = sm.OLS(contemp_df["y"], contemp_X).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})

    lag_cols = [f"K_lag{lag}" for lag in range(1, n_lags + 1)]
    lag_df = df[["y"] + lag_cols].dropna()
    lag_X = sm.add_constant(lag_df[lag_cols])
    lag_model = sm.OLS(lag_df["y"], lag_X).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})
    return contemp_model, lag_model

def report(label, contemp_model, lag_model):
    print(f"=== {label} ===")
    print("-- Contemporaneous: y_t ~ const + K_t --")
    print(contemp_model.summary().tables[1])
    k_coef, k_p = contemp_model.params["K"], contemp_model.pvalues["K"]
    print(f"K coefficient: {k_coef:.4f} (p={k_p:.4f}, {'significant' if k_p < 0.05 else 'not significant'} at 5%)\n")
    print("-- Lagged (Melvin's addendum): y_t ~ const + K_(t-1..4) --")
    print(lag_model.summary().tables[1])
    print(f"Joint F-test on the 4 lags: F={lag_model.fvalue:.3f}, p={lag_model.f_pvalue:.4f}\n")


## 4 · Does widening K → consumer credit worsen?

In [5]:
contemp, lagged = contemporaneous_and_lagged_test(bottom50_leverage_chg, K)
report("Widening K -> consumer credit worsens? (Bottom 50% credit/assets leverage, QoQ change)", contemp, lagged)


=== Widening K -> consumer credit worsens? (Bottom 50% credit/assets leverage, QoQ change) ===
-- Contemporaneous: y_t ~ const + K_t --
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0002      0.001      0.230      0.818      -0.001       0.001
K              0.0020      0.001      1.864      0.062      -0.000       0.004
K coefficient: 0.0020 (p=0.0624, not significant at 5%)

-- Lagged (Melvin's addendum): y_t ~ const + K_(t-1..4) --
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0001      0.001      0.165      0.869      -0.001       0.002
K_lag1         0.0013      0.001      0.861      0.389      -0.002       0.004
K_lag2        -0.0008      0.002     -0.459      0.646      -0.004       0.002
K_lag3         0.0011      0.002     

## 5 · Does widening K → stocks go up?

In [6]:
contemp, lagged = contemporaneous_and_lagged_test(agg_equity_growth, K)
report("Widening K -> stocks go up? (DFA-derived aggregate equity growth proxy)", contemp, lagged)


=== Widening K -> stocks go up? (DFA-derived aggregate equity growth proxy) ===
-- Contemporaneous: y_t ~ const + K_t --
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0224      0.008      2.774      0.006       0.007       0.038
K              0.0174      0.011      1.633      0.102      -0.003       0.038
K coefficient: 0.0174 (p=0.1024, not significant at 5%)

-- Lagged (Melvin's addendum): y_t ~ const + K_(t-1..4) --
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0222      0.008      2.637      0.008       0.006       0.039
K_lag1         0.0073      0.025      0.290      0.772      -0.042       0.056
K_lag2         0.0050      0.032      0.155      0.877      -0.058       0.068
K_lag3         0.0309      0.038      0.818      0.4

**Reading these results:** both point the direction the advisor expects (positive sign in
both cases) but neither clears a conventional 5% bar (consumer credit: p=0.062 contemporaneous,
p=0.067 joint on lags — the closer call; equity growth: p=0.102 contemporaneous, p=0.708 on
lags — no lagged effect at all). This is consistent with the advisor's own framing: a
directionally-consistent-but-marginal contemporaneous relationship is close to what a
mechanical wealth-effect channel predicts using public, low-frequency data, and a strong,
easily-exploitable *lagged* edge would be the surprising result — if quarterly public data
reliably predicted stock returns a quarter ahead, that edge would likely already be
arbitraged away (Buffett's point).

## 6 · The advisor's asset-price/econ-growth regressions

`variable_t ~ const + K_t + K_(t-1..4)`, HAC errors, for: a stock index, 10Y Treasury total
return, USD/JPY, USD/EUR, USD/GBP, gold, unemployment rate, GDP, and industrial production.

**These need real market/macro data this environment cannot fetch** — Yahoo Finance, FRED,
Stooq, Alpha Vantage, an exchange-rate API, BLS, and BEA are all blocked by this sandbox's
network policy (confirmed directly via `curl`, not assumed). Drop CSVs (columns `Date,Value`)
into `data_cache/<name>.csv` to run these for real — see the table below for exactly which
files and suggested sources. Until then, each target is skipped with a clear message instead
of a fabricated result.

In [7]:
SERIES_FILES = {
    "sp500": "sp500.csv", "treasury_10y_total_return": "treasury_10y_total_return.csv",
    "usdjpy": "usdjpy.csv", "usdeur": "usdeur.csv", "usdgbp": "usdgbp.csv", "gold": "gold.csv",
    "unemployment_rate": "unemployment_rate.csv", "gdp": "gdp.csv",
    "industrial_production": "industrial_production.csv",
}
ASSET_PRICE_TARGETS = {"sp500": "pct_change", "treasury_10y_total_return": "pct_change",
                       "usdjpy": "pct_change", "usdeur": "pct_change", "usdgbp": "pct_change", "gold": "pct_change"}
ECON_GROWTH_TARGETS = {"unemployment_rate": "diff", "gdp": "pct_change", "industrial_production": "pct_change"}

def load_target(name, transform):
    path = CACHE_DIR / SERIES_FILES[name]
    if not path.exists():
        raise RuntimeError(
            f"Missing {path}. This sandbox cannot fetch '{name}' live (every market/macro data "
            "host tried is blocked by network policy) -- drop a CSV with columns Date,Value there to proceed."
        )
    level = pd.read_csv(path, parse_dates=["Date"]).set_index("Date").sort_index()["Value"].resample("QE").last()
    return (level.pct_change() if transform == "pct_change" else level.diff()).rename(name)

def run_k_regression(target, k, n_lags=4, hac_lags=3):
    df = pd.DataFrame({"K": k})
    for lag in range(1, n_lags + 1):
        df[f"K_lag{lag}"] = df["K"].shift(lag)
    df = pd.concat([target.rename("y"), df], axis=1).dropna()
    X = sm.add_constant(df.drop(columns="y"))
    return sm.OLS(df["y"], X).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags}), df

results = {}
for group_name, targets in [("1. Does K explain asset prices?", ASSET_PRICE_TARGETS),
                             ("2. Does K explain economic growth?", ECON_GROWTH_TARGETS)]:
    print(f"=== {group_name} ===\n")
    for name, transform in targets.items():
        try:
            target = load_target(name, transform)
        except RuntimeError as e:
            print(f"[SKIPPED] {name}: {e}\n")
            continue
        model, df = run_k_regression(target, K)
        results[name] = model
        print(f"--- {name} (n={len(df)}) ---")
        print(model.summary().tables[1])
        print()

if not results:
    print("No target data available yet in this session -- every regression was skipped. "
          "Add CSVs to data_cache/ (see table above) and re-run this cell.")


=== 1. Does K explain asset prices? ===

[SKIPPED] sp500: Missing data_cache/sp500.csv. This sandbox cannot fetch 'sp500' live (every market/macro data host tried is blocked by network policy) -- drop a CSV with columns Date,Value there to proceed.

[SKIPPED] treasury_10y_total_return: Missing data_cache/treasury_10y_total_return.csv. This sandbox cannot fetch 'treasury_10y_total_return' live (every market/macro data host tried is blocked by network policy) -- drop a CSV with columns Date,Value there to proceed.

[SKIPPED] usdjpy: Missing data_cache/usdjpy.csv. This sandbox cannot fetch 'usdjpy' live (every market/macro data host tried is blocked by network policy) -- drop a CSV with columns Date,Value there to proceed.

[SKIPPED] usdeur: Missing data_cache/usdeur.csv. This sandbox cannot fetch 'usdeur' live (every market/macro data host tried is blocked by network policy) -- drop a CSV with columns Date,Value there to proceed.

[SKIPPED] usdgbp: Missing data_cache/usdgbp.csv. This san

## Summary

- **Consumer credit and equity-growth mechanism tests** (real data, computed above): both
  point the hypothesized direction, neither clears 5% significance — a marginal, honest
  result, not a failure.
- **Asset-price/econ-growth regressions**: code is complete and correct; waiting on real
  market/macro data (see the file table above for exactly what's needed).
- **Next step, if pursued:** Pillar 4 of the original deck already cites the NY Fed Consumer
  Credit Panel / Equifax data for its mortgage-FICO chart — that (or its delinquency-rate cut)
  would be a more direct "consumer credit worsens" measure than the leverage-ratio proxy used
  here, if it's available to the team.